In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import QuantileTransformer, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

# Understanding and Getting The Data

In [ ]:
train_data = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test_data = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

print("Train data shape: ", train_data.shape)
print("Test data shape: ", test_data.shape)
print(train_data.info())
print(train_data.isnull().sum())

# Looking at Heatmap

In [ ]:
corr_matrix = train_data.select_dtypes(include=['number']).corr()
plt.figure(figsize=(17, 15)) 
sns.heatmap(
    corr_matrix, 
    annot=False,     
    cmap='coolwarm',  
    fmt='.2f'       
)
plt.title('Correlation Heatmap of Numerical Features')
plt.show()

# Splitting Data to Train

In [ ]:
X_full = train_data.drop(columns='SalePrice')
y_full = train_data['SalePrice']
y_full_log = np.log1p(y_full)
X_train_orig, X_val_orig, y_train_log, y_val_log = train_test_split(
    X_full, y_full_log, test_size=0.2, random_state=42
)

# Feature Engineering
1. Feature engineer some that would need it and make list of each column that were being engineered 
2. Split the data into two types: \
a. Numerical \
b. Categorial, but also also divided again into two, (1) ordinal and (2) one-hot-encoded \
c. The one that doesn't need any more feature engineering

In [ ]:
num_features = ['LotFrontage', 'LotArea', 'Age', 'AgeSinceRemod', 'MasVnrArea','TotalBsmtSF', 'TotalBsmtFinSF',
                'GrLivArea','TotalBath','TotRmsAbvGrd', 'Fireplaces','KitchenToRoomRatio', 'BedroomToRoomRatio',
                'BathPerRoom', 'GarageAge', 'AreaPerCar', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch',
                '3SsnPorch', 'ScreenPorch', 'PoolArea', 'MiscVal', 'GarageAge_x_AreaPerCar'
               ]

cat_ohe_features = ['MSSubClass', 'MSZoning', 'House_Access', 'LotConfig', 'Neighborhood',
                    'BldgType', 'HouseStyle', 'RoofMatl', 'RoofStyle', 'MasVnrType',
                    'Foundation', 'Heating', 'Electrical', 'GarageType', 'MiscFeature',
                    'SaleType', 'SaleCondition', 'MoSold', 'LandContour'
                   ]

# The order for each ordinal features
ordinal_order = {
'LotShape' : ['Reg', 'IR1', 'IR2', 'IR3'],
'Utilities' : ['ELO', 'NoSeWa', 'NoSewr', 'AllPub'],
'LandSlope' : ['Sev', 'Mod', 'Gtl'],
'ExterQual' : ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
'ExterCond' : ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
'BsmtQual' : ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
'BsmtCond' : ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
'BsmtExposure' : ['NA', 'No', 'Mn', 'Av', 'Gd'],
'BsmtFinType1' : ['NA', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
'BsmtFinType2' : ['NA', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
'HeatingQC' : ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
'KitchenQual' : ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
'Functional': ['Sal', 'Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ'],
'FireplaceQu' : ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
'GarageFinish' : ['NA', 'Unf', 'RFn', 'Fin'],
'GarageQual' : ['NA', 'Po', 'Fa', 'TA', 'Gd','Ex'],
'GarageCond' : ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
'PavedDrive' : ['N', 'P', 'Y'],
'PoolQC' : ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
'Fence' : ['NA', 'MnWw', 'GdWo', 'MnPrv', 'GdPrv']
}

cat_ord_features = list(ordinal_order.keys())
ordered_cat = [ordinal_order[col] for col in cat_ord_features]

def CombineTwoColSameData(df, cond1, cond2):
    cond_dummies = pd.get_dummies(df[[cond1, cond2]].stack()).groupby(level=0).max()
    df = pd.concat([df, cond_dummies], axis=1)
    cond_columns = cond_dummies.columns.tolist()
    return df, cond_columns
    
def FeatureEngineering(df_input):
    df = df_input.copy()
    pass_features = []
    feature_dropped = []
    # Change the type for MSSubClass from int to string
    df['MSSubClass'] = df['MSSubClass'].astype('string')

    # Integrate Street and Alley together into House Access
    df['Street'] = df['Street'].fillna('None')
    df['Alley'] = df['Alley'].fillna('None')
    df['House_Access'] = df['Street'] + '_' + df['Alley'].replace('None', '')
    feature_dropped.extend(['Street', 'Alley']) # There's only 5 x 4 combination so maybe it's fine

    # Make 9 features from condition1 and 2
    df, condition_columns= CombineTwoColSameData(df, 'Condition1', 'Condition2')
    pass_features.extend(condition_columns)
    feature_dropped.extend(['Condition1', 'Condition2'])

    # Let overallCond and OverallQual stay the same
    pass_features.extend(['OverallQual', 'OverallCond'])

    # Make three new features with YrBuilt, YrRemod, YrSold
    df['Age'] = df['YrSold'] - df['YearBuilt']
    df['AgeSinceRemod'] = df['YrSold'] - df['YearRemodAdd']
    df['WasRemod'] = (df['YearBuilt'] != df['YearRemodAdd']).astype(int)
    feature_dropped.extend(['YearBuilt', 'YearRemodAdd', 'YrSold'])
    pass_features.append('WasRemod')
    
    # Make more features with exterior
    df, exterior_columns = CombineTwoColSameData(df, 'Exterior1st', 'Exterior2nd')
    pass_features.extend(exterior_columns)
    feature_dropped.extend(['Exterior1st', 'Exterior2nd'])

    # Delete these two features as there's already a feature for it
    feature_dropped.extend(['BsmtFinSF1','BsmtFinSF2', 'BsmtUnfSF'])
    # Get the total FINISHED basement area (isi NaN dgn 0 dulu)
    cols_bsmt = ['BsmtFinSF1', 'BsmtFinSF2']
    df[cols_bsmt] = df[cols_bsmt].fillna(0)
    df['TotalBsmtFinSF'] = df['BsmtFinSF1'] + df['BsmtFinSF2']

    #Because central air is binary
    df['CentralAir'] = df['CentralAir'].map({'N' : 0, 'Y': 1})
    pass_features.append('CentralAir')

    # Drop these because it's the same as GrLivArea
    feature_dropped.extend(['1stFlrSF', '2ndFlrSF', 'LowQualFinSF'])
    
    

    # Get the total bathrooms
    cols = ['FullBath', 'BsmtFullBath', 'HalfBath', 'BsmtHalfBath']
    df[cols] = df[cols].fillna(0)
    df['TotalBath'] = df['FullBath'] + df['BsmtFullBath'] + (0.5 * df['HalfBath']) + (0.5 * df['BsmtHalfBath'])
    feature_dropped.extend(cols)

    # Make features: HasPool and HasFireplace
    df['HasPool'] = (df['PoolArea'] > 0).astype(int)
    df['HasFireplace'] = (df['Fireplaces'] > 0).astype(int)
    pass_features.extend(['HasPool', 'HasFireplace'])

    # Get ratio between rooms
    temp_tot_rms = df['TotRmsAbvGrd'].replace(0, np.nan) 
    df['BedroomToRoomRatio'] = df['BedroomAbvGr'] / temp_tot_rms
    df['KitchenToRoomRatio'] = df['KitchenAbvGr'] / temp_tot_rms
    df['BathPerRoom'] = df['TotalBath'] / temp_tot_rms
    feature_dropped.extend(['BedroomAbvGr', 'KitchenAbvGr'])

    #GarageAge : Age of Garage since house been built
    df['GarageAge'] = df['GarageYrBlt'] - df['YearBuilt']
    
    # Get Area per cats and replace 0 cars with 1 
    df['AreaPerCar'] = df['GarageArea'] / df['GarageCars'].replace(0, 1)
    feature_dropped.extend(['GarageArea', 'GarageCars'])

    # Feature interaction between GarageAge and AreaPerCar
    df['GarageAge_x_AreaPerCar'] = df['GarageAge'] * df['AreaPerCar']
    
    # Porch Area
    cols = ['WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch']
    df[cols] = df[cols].fillna(0)
    df['HasPorch'] = (df['WoodDeckSF'] + df['OpenPorchSF'] + df['EnclosedPorch'] + df['3SsnPorch'] + df['ScreenPorch'] > 0).astype(int)
    pass_features.append('HasPorch')

    # Fill some features with 0
    cols = ['Fireplaces', 'PoolArea', 'MasVnrArea', 'MiscVal', 'BedroomToRoomRatio', 'KitchenToRoomRatio', 'BathPerRoom']
    df[cols] = df[cols].fillna(0)
    return df, pass_features, feature_dropped
    

print("Running Feature Engineering...")
X_train_fe, passthrough_features, drop_features = FeatureEngineering(X_train_orig)
X_val_fe, _, _ = FeatureEngineering(X_val_orig)
X_train_fe = X_train_fe.drop(drop_features, axis=1)
X_val_fe = X_val_fe.drop(drop_features, axis=1)
passthrough_features = list(set(passthrough_features))
master_cols = X_train_fe.columns
X_val_fe = X_val_fe.reindex(columns=master_cols, fill_value=0)
print("Feature Engineering Finished.")

# Heatmap After Feature Engineering

In [ ]:
corr_matrix = X_train_fe.select_dtypes(include=['number']).corr()
plt.figure(figsize=(17, 15)) 
sns.heatmap(
    corr_matrix, 
    annot=False,     
    cmap='coolwarm',  
    fmt='.2f'       
)
plt.title('Correlation Heatmap of Numerical Features')
plt.show()

# Preprocessor Pipeline

In [ ]:
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), 
    ('quantile', QuantileTransformer(output_distribution='normal', random_state=42, n_quantiles=900))
])

cat_ohe_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])


cat_ord_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), 
    ('encoder', OrdinalEncoder(categories=ordered_cat))
])

pass_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')) 
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, num_features),
        ('cat_ohe', cat_ohe_pipeline, cat_ohe_features),
        ('cat_ord', cat_ord_pipeline, cat_ord_features),
        ('pass', pass_pipeline, passthrough_features)
    ],
    remainder='drop' 
)

# Model Pipeline

In [ ]:
model = XGBRegressor(
    objective='reg:squarederror',  # regression task
    random_state=42,
    n_jobs=-1,                     # use all CPU cores
)

full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', model)
])

param_distributions = {
    'model__n_estimators': [300, 500, 800, 1000],     # number of boosting rounds (trees)
    'model__learning_rate': [0.01, 0.05, 0.1, 0.2],   # shrinkage rate; smaller = slower but better
    'model__max_depth': [3, 4, 5, 6, 8, 10],          # depth of each tree
    'model__min_child_weight': [1, 3, 5, 7],          # min sum of instance weight in a leaf
    'model__subsample': [0.6, 0.8, 1.0],              # row sampling (like max_samples)
    'model__colsample_bytree': [0.5, 0.7, 0.8, 1.0],  # feature sampling per tree
    'model__gamma': [0, 0.1, 0.2, 0.3],               # min loss reduction required for further split
    'model__reg_alpha': [0, 0.1, 0.5, 1.0],           # L1 regularization
    'model__reg_lambda': [0.5, 1.0, 1.5, 2.0]         # L2 regularization
}


random_search = RandomizedSearchCV(
    estimator=full_pipeline,
    param_distributions=param_distributions,
    n_iter=25, 
    cv=5,      
    verbose=2, 
    random_state=42,
    n_jobs=-1,
    scoring='neg_root_mean_squared_error' 
)

# Training with The Model

In [ ]:
print("Starting RandomizedSearchCV (hyperparameter tuning)...")
random_search.fit(X_train_fe, y_train_log)
print("Tuning Finished.")


print("\n--- EVALUATION RESULT ---")
print(f"Best Parameter: {random_search.best_params_}")


train_score = random_search.score(X_train_fe, y_train_log)
print(f"RMSE: {train_score:.4f}")


val_score = random_search.score(X_val_fe, y_val_log)
print(f"RMSE on Data Validation: {val_score:.4f}")

# Predicting

In [ ]:
test_data = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')
test_ids = test_data['Id']
X_test_final = test_data.copy()
X_test_fe, _, _ = FeatureEngineering(X_test_final)
X_test_fe = X_test_fe.reindex(columns=master_cols, fill_value=0)
print("\nMembuat prediksi pada data test...")

log_predictions = random_search.predict(X_test_fe)
final_predictions = np.expm1(log_predictions)

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': final_predictions
})

submission['SalePrice'] = submission['SalePrice'].apply(lambda x: max(0, x))

submission.to_csv('submission.csv', index=False)
print("File submission.csv has been made.")
submission.head()